# 08 - Consolidate MLP Ternary Results

Consolida os resultados da fase MLP ternária e gera uma tabela comparável para decidir próximos benchmarks.

In [ ]:
from pathlib import Path
import json
import csv

PROJECT_NAME = 'gemma4_theory'
COLAB_DRIVE_ROOT = Path('/content/drive/MyDrive/RPT')

candidates = []
if COLAB_DRIVE_ROOT.exists():
    candidates.append(COLAB_DRIVE_ROOT / PROJECT_NAME)
candidates.append(Path.cwd() / PROJECT_NAME)
candidates.append(Path.cwd())

PROJECT_ROOT = None
for candidate in candidates:
    if (candidate / 'results').exists() or (candidate / 'data').exists() or candidate.name == PROJECT_NAME:
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd() / PROJECT_NAME
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = PROJECT_ROOT / 'results'
OUT_DIR = RESULTS_DIR / 'consolidated'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)
print('OUT_DIR =', OUT_DIR)


In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

rows = []

# Fixed baseline/result files already produced in this track.
known = [
    ('baseline', RESULTS_DIR / '01_baseline_eval_20260420_060326.json'),
    ('pruned_checkpoint', RESULTS_DIR / '03_pruned_checkpoint_eval_20260420_065514.json'),
    ('quantization', RESULTS_DIR / '04_quantization_eval_20260420_073000.json'),
]

for label, path in known:
    if not path.exists():
        continue
    data = load_json(path)
    if isinstance(data, list):
        for item in data:
            rows.append({
                'stage': item.get('variant', label),
                'target_layer_limit': '',
                'pre_recovery_accuracy': '',
                'post_recovery_accuracy': item.get('summary', {}).get('accuracy'),
                'source_file': str(path),
            })
    else:
        rows.append({
            'stage': label,
            'target_layer_limit': '',
            'pre_recovery_accuracy': '',
            'post_recovery_accuracy': data.get('summary', data).get('accuracy'),
            'source_file': str(path),
        })

for path in sorted(RESULTS_DIR.glob('**/07_ternary_expansion_summary_*.json')):
    data = load_json(path)
    for stage in data.get('stages', []):
        rows.append({
            'stage': 'ternary_mlp_recovery',
            'target_layer_limit': stage.get('target_layer_limit'),
            'pre_recovery_accuracy': stage.get('pre_recovery_accuracy'),
            'post_recovery_accuracy': stage.get('post_recovery_accuracy'),
            'source_file': str(path),
        })

# Deduplicate by stage + target + post accuracy + source file.
seen = set()
deduped = []
for row in rows:
    key = tuple(row.items())
    if key in seen:
        continue
    seen.add(key)
    deduped.append(row)

deduped = sorted(deduped, key=lambda r: (str(r['stage']), int(r['target_layer_limit'] or -1), str(r['source_file'])))

print('rows:', len(deduped))
for row in deduped:
    print(row)


In [ ]:
csv_path = OUT_DIR / 'mlp_ternary_summary.csv'
json_path = OUT_DIR / 'mlp_ternary_summary.json'

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['stage', 'target_layer_limit', 'pre_recovery_accuracy', 'post_recovery_accuracy', 'source_file'])
    writer.writeheader()
    writer.writerows(deduped)

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(deduped, f, ensure_ascii=False, indent=2)

print('CSV:', csv_path)
print('JSON:', json_path)


## Próximo passo

Se `78` manteve `0.91+`, a próxima fase deve ser benchmark externo: `lm-evaluation-harness` com HellaSwag/ARC/MMLU-lite/TruthfulQA, antes de mexer em attention.